# Explore FooDB — Flavor Dataset

**Mission**: Data exploratory on the FooDB flavor dataset stored in `/workspace/ICOCO.duckdb`, using **only** the silver layer (`silver.FooDB_flavor`).

**Source**: built by [`../1_silver_create/silver_curation.ipynb`](../1_silver_create/silver_curation.ipynb) from the bronze `FooDB.flavor` table (kept columns `id, name, flavor_group, category`).

**Principles**
- **SQL-first** — every question answered with DuckDB SQL.
- **Read-only** — no tables/views/schemas are created or modified.
- **Silver only** — the bronze `FooDB.*` tables are *not* touched; we only read the curated `silver.FooDB_flavor`.

## 0. Setup (connect to DuckDB)

Follows the same connection pattern as the silver notebook. Only **one** DuckDB connection is open at a time (DuckDB limitation).

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100  # show all rows in query results
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
# Close the connection first to flush the WAL file (same pattern as silver notebook).
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL)
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

## 1. Overview — what is in the table

`silver.FooDB_flavor` stores **flavor descriptors**: an `id`, a `name` (the flavor word), a `flavor_group` (a coarse taste/odor family) and a `category` (the perception type).

In [4]:
%%sql
DESCRIBE silver.FooDB_flavor;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

column_name,column_type,null,key,default,extra


Sample rows:

In [5]:
%%sql
SELECT * FROM silver.FooDB_flavor LIMIT 12;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

id,name,flavor_group,category
1,celery,vegetable,odor
2,corn,vegetable,odor
3,cucumber,vegetable,odor
4,horseradish,vegetable,odor
5,vegetable,vegetable,odor
6,potato,vegetable,odor
7,tomato,vegetable,odor
8,caraway,herbaceous,odor
9,clove,herbaceous,odor
10,fennel,herbaceous,odor


## 2. Rows & data quality (nulls)

How many flavor descriptors do we have, and how complete is each column?

In [6]:
%%sql
SELECT COUNT(*)                       AS n_rows,
       COUNT(name)                     AS n_name,
       COUNT(flavor_group)             AS n_flavor_group,
       COUNT(category)                 AS n_category
FROM silver.FooDB_flavor;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

n_rows,n_name,n_flavor_group,n_category
883,883,113,883


**Read**: all 883 entries have a `name` and a `category`, but `flavor_group` is missing for the majority.

## 3. Category distribution

The `category` column tells us how the flavor is perceived (odor / taste / …).

In [7]:
%%sql
SELECT category,
       COUNT(*) AS n
FROM silver.FooDB_flavor
GROUP BY category
ORDER BY n DESC;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

category,n
odor,883


**Read**: the curated silver extract contains **only `odor`** descriptors (all 883). So this table is an **odor-lexicon** — no taste/taste-and-odor categories survived the silver curation.

## 4. Flavor group distribution

`flavor_group` is a coarse family (e.g. `fruity`, `floral`, `balsamic`). How are the 883 descriptors distributed across groups?

In [8]:
%%sql
SELECT COALESCE(flavor_group, '(none / NULL)') AS flavor_group,
       COUNT(*)                            AS n
FROM silver.FooDB_flavor
GROUP BY flavor_group
ORDER BY n DESC;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

flavor_group,n
(none / NULL),770
fruity,24
floral,14
balsamic,10
vegetable,8
fatty,7
herbaceous,5
nutty,5
citrus,5
wine_like,5


## 5. Category × flavor group

Since every row is `category = odor`, the only useful cross-tab here is the group sizes; we show the entries inside a few representative groups to see what the words look like.

In [9]:
%%sql
SELECT name, flavor_group, category
FROM silver.FooDB_flavor
WHERE flavor_group IS NOT NULL
ORDER BY flavor_group, name
LIMIT 60;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

name,flavor_group,category
animal,animal,odor
anise,balsamic,odor
balsam,balsamic,odor
butterscotch,balsamic,odor
caramel,balsamic,odor
chocolate,balsamic,odor
cinnamon,balsamic,odor
honey,balsamic,odor
others,balsamic,odor
sweet,balsamic,odor


## 6. Summary / findings worth investigating

| Question | Answer |
|----------|--------|
| Total flavor descriptors | **883** |
| Column completeness | `name` & `category` 100% complete; `flavor_group` only ~13% complete |
| Distinct categories | **1** — all rows are `odor` |
| Distinct flavor groups | **27** (plus `NULL` for 770 rows) |
| Largest flavor group | `fruity` (24), `floral` (14), `balsamic` (10), `vegetable` (8), `fatty` (7) |
| Mini-groups | many singleton groups (12 groups have only 1 member) |

**Worth investigating**
- **~87% of descriptors have no `flavor_group`** (770 / 883). A downstream user joining on `flavor_group` would silently drop most entries — the group taxonomy is very sparse.
- **Only `odor` survives** in `silver.FooDB_flavor`. If the full bronze `FooDB.flavor` includes `taste` / `taste_and_odor` categories, the silver curation may have been lossy — worth checking before relying on it as "the FooDB flavor table".
- The **group sizes are extremely unbalanced** (fruity=24 vs smoky/minty/…=1), which is normal for a lexicon but means any analysis must expect heavy imbalance.